In [ ]:
import numpy, matplotlib, rasterio, pystac_client, planetary_computer, stackstac, rich
print("All good")

ModuleNotFoundError: No module named 'pystac_client'

In [ ]:
import pystac_client
import planetary_computer
import stackstac
import numpy as np

# Lincolnshire farmland bounding box (lon_min, lat_min, lon_max, lat_max)
BBOX = [-0.20, 52.90, -0.05, 53.05]

# Microsoft's Planetary Computer for Sentinel-2 imagery
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime="2024-06-01/2024-08-31",  # Summer
    query={"eo:cloud_cover": {"lt": 10}}  # Less than 10% cloud cover
)

items = list(search.items())
print(f"Found {len(items)} usable Sentinel-2 scenes")
print(f"Using scene from: {items[0].datetime.strftime('%d %B %Y')}")
print(f"Cloud cover: {items[0].properties['eo:cloud_cover']:.1f}%")

In [ ]:
import stackstac

# lowest cloud cover comes first
item = items[0]

# Load only the bands we need
# B04 = Red, B08 = Near-Infrared (NIR)
stack = stackstac.stack(
    [item],
    assets=["B04", "B08"],
    resolution=20,
    bounds_latlon=BBOX,
    epsg=3857
)

red = stack.sel(band="B04").values[0]
nir = stack.sel(band="B08").values[0]

print(f"Image loaded successfully")
print(f"Grid size: {red.shape[0]} x {red.shape[1]} pixels")
print(f"Each pixel covers: 20m x 20m of real farmland")
print(f"Total area covered: ~{round((red.shape[0]*20 * red.shape[1]*20)/10000)} hectares")

In [ ]:
import numpy as np


red = red.astype(float)
nir = nir.astype(float)

with np.errstate(divide='ignore', invalid='ignore'):
    ndvi = np.where(
        (nir + red) == 0,
        np.nan,
        (nir - red) / (nir + red)
    )


ndvi[ndvi < -1] = np.nan
ndvi[ndvi > 1] = np.nan


valid = ndvi[~np.isnan(ndvi)]
print(f"NDVI calculated across {len(valid):,} pixels of farmland")
print(f"")
print(f"NDVI Statistics:")
print(f"  Min:    {np.nanmin(ndvi):.3f}  (bare soil / water)")
print(f"  Max:    {np.nanmax(ndvi):.3f}  (dense healthy crop)")
print(f"  Mean:   {np.nanmean(ndvi):.3f}")
print(f"  Median: {np.nanmedian(ndvi):.3f}")
print(f"")
print(f"Crop Health Breakdown:")
print(f"  Healthy  (NDVI > 0.4):  {np.sum(valid > 0.4):,} pixels  ({np.sum(valid > 0.4)/len(valid)*100:.1f}%)")
print(f"  Stressed (0.2-0.4):     {np.sum((valid >= 0.2) & (valid <= 0.4)):,} pixels  ({np.sum((valid >= 0.2) & (valid <= 0.4))/len(valid)*100:.1f}%)")
print(f"  Critical (< 0.2):       {np.sum(valid < 0.2):,} pixels  ({np.sum(valid < 0.2)/len(valid)*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#0a0f0a')


ax1 = axes[0]
ax1.set_facecolor('#0a0f0a')

ndvi_plot = ax1.imshow(
    ndvi,
    cmap='RdYlGn',
    vmin=-0.2,
    vmax=0.8,
    interpolation='nearest'
)
cbar = plt.colorbar(ndvi_plot, ax=ax1, fraction=0.03, pad=0.04)
cbar.set_label('NDVI Value', color='#c8e6c8', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#c8e6c8')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#c8e6c8', fontsize=8)

ax1.set_title('NDVI — Lincolnshire Farmland\n23 August 2024 · Sentinel-2',
              color='#2dff6e', fontsize=11, pad=12, fontweight='bold')
ax1.set_xlabel('Pixel Column (1px = 20m)', color='#5a7a5a', fontsize=8)
ax1.set_ylabel('Pixel Row (1px = 20m)', color='#5a7a5a', fontsize=8)
ax1.tick_params(colors='#5a7a5a', labelsize=7)
for spine in ax1.spines.values():
    spine.set_edgecolor('#1a3a1a')


ax2 = axes[1]
ax2.set_facecolor('#0a0f0a')


classified = np.full(ndvi.shape, np.nan)
classified[ndvi > 0.4] = 2
classified[(ndvi >= 0.2) & (ndvi <= 0.4)] = 1
classified[ndvi < 0.2] = 0
cmap_class = mcolors.ListedColormap(['#4d0a0a', '#4d3d00', '#1a4d1a'])
bounds_class = [-0.5, 0.5, 1.5, 2.5]
norm_class = mcolors.BoundaryNorm(bounds_class, cmap_class.N)

ax2.imshow(
    classified,
    cmap=cmap_class,
    norm=norm_class,
    interpolation='nearest'
)

legend_elements = [
    Patch(facecolor='#1a4d1a', label=f'Healthy   (NDVI > 0.4) — 22.8%'),
    Patch(facecolor='#4d3d00', label=f'Stressed  (NDVI 0.2–0.4) — 29.2%'),
    Patch(facecolor='#4d0a0a', label=f'Critical   (NDVI < 0.2) — 48.0%'),
]
ax2.legend(handles=legend_elements, loc='lower right',
           facecolor='#0f1a0f', edgecolor='#1a3a1a',
           labelcolor='#c8e6c8', fontsize=8)

ax2.set_title('Classified Crop Health Map\nDrone AI Detection Simulation',
              color='#2dff6e', fontsize=11, pad=12, fontweight='bold')
ax2.set_xlabel('Pixel Column (1px = 20m)', color='#5a7a5a', fontsize=8)
ax2.set_ylabel('Pixel Row (1px = 20m)', color='#5a7a5a', fontsize=8)
ax2.tick_params(colors='#5a7a5a', labelsize=7)
for spine in ax2.spines.values():
    spine.set_edgecolor('#1a3a1a')

plt.tight_layout()
plt.savefig('ndvi_lincolnshire.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0f0a')
plt.show()
print("Saved as ndvi_lincolnshire.png")

In [ ]:

valid_cols = np.where(~np.all(np.isnan(ndvi), axis=0))[0]
valid_rows = np.where(~np.all(np.isnan(ndvi), axis=1))[0]

col_start, col_end = valid_cols[0], valid_cols[-1]
row_start, row_end = valid_rows[0], valid_rows[-1]

ndvi_crop = ndvi[row_start:row_end, col_start:col_end]
classified_crop = classified[row_start:row_end, col_start:col_end]

print(f"Cropped to actual data: {ndvi_crop.shape[0]} x {ndvi_crop.shape[1]} pixels")
print(f"Real area shown: ~{round((ndvi_crop.shape[0]*20 * ndvi_crop.shape[1]*20)/10000):,} hectares")


fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.patch.set_facecolor('#0a0f0a')


ax1 = axes[0]
ax1.set_facecolor('#0a0f0a')
ndvi_plot = ax1.imshow(ndvi_crop, cmap='RdYlGn', vmin=-0.2, vmax=0.8, interpolation='nearest')
cbar = plt.colorbar(ndvi_plot, ax=ax1, fraction=0.03, pad=0.04)
cbar.set_label('NDVI Value', color='#c8e6c8', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='#c8e6c8')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#c8e6c8', fontsize=8)
ax1.set_title('NDVI — Lincolnshire Farmland\n23 August 2024 · Sentinel-2',
              color='#2dff6e', fontsize=11, pad=12, fontweight='bold')
ax1.set_xlabel('Pixel Column (1px = 20m)', color='#5a7a5a', fontsize=8)
ax1.set_ylabel('Pixel Row (1px = 20m)', color='#5a7a5a', fontsize=8)
ax1.tick_params(colors='#5a7a5a', labelsize=7)
for spine in ax1.spines.values():
    spine.set_edgecolor('#1a3a1a')


ax2 = axes[1]
ax2.set_facecolor('#0a0f0a')
cmap_class = mcolors.ListedColormap(['#4d0a0a', '#4d3d00', '#1a4d1a'])
bounds_class = [-0.5, 0.5, 1.5, 2.5]
norm_class = mcolors.BoundaryNorm(bounds_class, cmap_class.N)
ax2.imshow(classified_crop, cmap=cmap_class, norm=norm_class, interpolation='nearest')

legend_elements = [
    Patch(facecolor='#1a4d1a', label=f'Healthy   (NDVI > 0.4) — 22.8%'),
    Patch(facecolor='#4d3d00', label=f'Stressed  (NDVI 0.2–0.4) — 29.2%'),
    Patch(facecolor='#4d0a0a', label=f'Critical   (NDVI < 0.2) — 48.0%'),
]
ax2.legend(handles=legend_elements, loc='lower right',
           facecolor='#0f1a0f', edgecolor='#1a3a1a',
           labelcolor='#c8e6c8', fontsize=8)
ax2.set_title('Classified Crop Health Map\nDrone AI Detection Simulation',
              color='#2dff6e', fontsize=11, pad=12, fontweight='bold')
ax2.set_xlabel('Pixel Column (1px = 20m)', color='#5a7a5a', fontsize=8)
ax2.set_ylabel('Pixel Row (1px = 20m)', color='#5a7a5a', fontsize=8)
ax2.tick_params(colors='#5a7a5a', labelsize=7)
for spine in ax2.spines.values():
    spine.set_edgecolor('#1a3a1a')

plt.tight_layout()
plt.savefig('ndvi_lincolnshire.png', dpi=150, bbox_inches='tight', facecolor='#0a0f0a')
plt.show()
print("Saved.")

In [ ]:
from rich.console import Console
from rich.table import Table

console = Console()


scenarios = {
    "5G Urban":    {"latency_ms": 8,    "reliability": 99.9, "cost_ha": 18,  "bandwidth_mbps": 1000, "viable": True},
    "4G Rural":    {"latency_ms": 80,   "reliability": 87.0, "cost_ha": 12,  "bandwidth_mbps": 20,   "viable": True},
    "No Signal":   {"latency_ms": None, "reliability": 0,    "cost_ha": None,"bandwidth_mbps": 0,    "viable": False},
}


requirements = {
    "max_latency_ms": 100,
    "min_reliability": 95.0,
    "min_bandwidth_mbps": 10,
}


total_pixels = 474022
stressed_pixels = 138516
critical_pixels = 227309
pixel_area_ha = (20 * 20) / 10000

stressed_ha = stressed_pixels * pixel_area_ha
critical_ha = critical_pixels * pixel_area_ha

print("=" * 60)
print("  NDVI ANALYSIS — REAL WORLD FINDINGS")
print("=" * 60)
print(f"  Study Area:        Central Lincolnshire, UK")
print(f"  Image Date:        23 August 2024")
print(f"  Data Source:       Sentinel-2 L2A (ESA / Microsoft)")
print(f"  Resolution:        20m per pixel")
print(f"  Total Area:        {total_pixels * pixel_area_ha:,.0f} ha analysed")
print(f"")
print(f"  Healthy Zones:     22.8%  ({474022*0.228*pixel_area_ha:,.0f} ha)")
print(f"  Stressed Zones:    29.2%  ({stressed_ha:,.0f} ha)")
print(f"  Critical Zones:    48.0%  ({critical_ha:,.0f} ha)*")
print(f"")
print(f"  * Note: High critical % reflects post-harvest bare soil,")
print(f"    typical for Lincolnshire arable land in late August.")
print(f"    Demonstrates importance of contextual AI interpretation.")
print("=" * 60)

print("\n")

# Build scenario table
table = Table(title="Connectivity Scenario Analysis — 5G Drone System",
              style="green", header_style="bold green")

table.add_column("Scenario",        style="white",  width=14)
table.add_column("Latency",         style="cyan",   width=10)
table.add_column("Reliability",     style="cyan",   width=12)
table.add_column("Bandwidth",       style="cyan",   width=12)
table.add_column("Cost/ha",         style="cyan",   width=10)
table.add_column("Real-Time Alert", style="cyan",   width=14)
table.add_column("Verdict",         style="bold",   width=18)

for name, s in scenarios.items():
    lat    = f"{s['latency_ms']}ms"   if s['latency_ms'] else "N/A"
    rel    = f"{s['reliability']}%"
    bw     = f"{s['bandwidth_mbps']} Mbps"
    cost   = f"£{s['cost_ha']}/ha"   if s['cost_ha'] else "N/A"


    lat_ok  = s['latency_ms'] is not None and s['latency_ms'] <= requirements['max_latency_ms']
    rel_ok  = s['reliability'] >= requirements['min_reliability']
    bw_ok   = s['bandwidth_mbps'] >= requirements['min_bandwidth_mbps']

    alert   = "✓ Yes" if (lat_ok and bw_ok) else "✗ No"

    if lat_ok and rel_ok and bw_ok:
        verdict = "[green]ADOPT[/green]"
    elif lat_ok or bw_ok:
        verdict = "[yellow]CONDITIONAL[/yellow]"
    else:
        verdict = "[red]REJECT[/red]"

    table.add_row(name, lat, rel, bw, cost, alert, verdict)

console.print(table)

print("\n")
print("=" * 60)
print("  STRATEGIC RECOMMENDATION SUMMARY")
print("=" * 60)
print("  5G Urban:  ADOPT — meets all operational requirements.")
print("  4G Rural:  CONDITIONAL — viable for alerts but reliability")
print("             below 95% threshold. Recommend hybrid edge cache.")
print("  No Signal: REJECT — real-time alerting not achievable.")
print("             Manual drone retrieval required, negating ROI.")
print("=" * 60)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.patch.set_facecolor('#0a0f0a')
fig.suptitle('Connectivity Scenario Analysis — 5G Drone Precision Agriculture System',
             color='#2dff6e', fontsize=13, fontweight='bold', y=1.02)

scenarios = ['5G Urban', '4G Rural', 'No Signal']
colors    = ['#2dff6e', '#f5e642', '#ff3d3d']


ax1 = axes[0]
ax1.set_facecolor('#0f1a0f')
latencies = [8, 80, 0]
bars = ax1.bar(scenarios, latencies, color=colors, width=0.5, edgecolor='none')
ax1.axhline(y=100, color='#ff3d3d', linestyle='--', linewidth=1.2, label='Max acceptable (100ms)')
ax1.set_title('End-to-End Latency', color='#c8e6c8', fontsize=10, pad=10)
ax1.set_ylabel('Milliseconds (ms)', color='#5a7a5a', fontsize=9)
ax1.set_ylim(0, 130)
ax1.tick_params(colors='#5a7a5a', labelsize=8)
ax1.legend(fontsize=7, facecolor='#0a0f0a', edgecolor='#1a3a1a', labelcolor='#c8e6c8')
for spine in ax1.spines.values(): spine.set_edgecolor('#1a3a1a')
ax1.set_facecolor('#0a0f0a')

for bar, val in zip(bars, latencies):
    label = f'{val}ms' if val > 0 else 'N/A'
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             label, ha='center', va='bottom', color='#c8e6c8', fontsize=9, fontweight='bold')

ax2 = axes[1]
ax2.set_facecolor('#0a0f0a')
reliability = [99.9, 87.0, 0]
bars2 = ax2.bar(scenarios, reliability, color=colors, width=0.5, edgecolor='none')
ax2.axhline(y=95, color='#ff3d3d', linestyle='--', linewidth=1.2, label='Min required (95%)')
ax2.set_title('Network Reliability', color='#c8e6c8', fontsize=10, pad=10)
ax2.set_ylabel('Uptime (%)', color='#5a7a5a', fontsize=9)
ax2.set_ylim(0, 115)
ax2.tick_params(colors='#5a7a5a', labelsize=8)
ax2.legend(fontsize=7, facecolor='#0a0f0a', edgecolor='#1a3a1a', labelcolor='#c8e6c8')
for spine in ax2.spines.values(): spine.set_edgecolor('#1a3a1a')
for bar, val in zip(bars2, reliability):
    label = f'{val}%' if val > 0 else 'N/A'
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             label, ha='center', va='bottom', color='#c8e6c8', fontsize=9, fontweight='bold')


ax3 = axes[2]
ax3.set_facecolor('#0a0f0a')
costs = [18, 12, 0]
bars3 = ax3.bar(scenarios, costs, color=colors, width=0.5, edgecolor='none')
ax3.set_title('Estimated Cost per Hectare', color='#c8e6c8', fontsize=10, pad=10)
ax3.set_ylabel('GBP (£) per hectare/season', color='#5a7a5a', fontsize=9)
ax3.set_ylim(0, 28)
ax3.tick_params(colors='#5a7a5a', labelsize=8)
for spine in ax3.spines.values(): spine.set_edgecolor('#1a3a1a')
for bar, val in zip(bars3, costs):
    label = f'£{val}' if val > 0 else 'N/A'
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             label, ha='center', va='bottom', color='#c8e6c8', fontsize=9, fontweight='bold')


verdicts = ['ADOPT', 'CONDITIONAL', 'REJECT']
verdict_colors = ['#2dff6e', '#f5e642', '#ff3d3d']
for ax, verdict, vc in zip(axes, verdicts, verdict_colors):
    ax.text(0.5, -0.18, verdict, transform=ax.transAxes,
            ha='center', fontsize=10, fontweight='bold', color=vc,
            fontfamily='monospace')

plt.tight_layout()
plt.savefig('scenario_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0a0f0a')
plt.show()
print("Saved as scenario_analysis.png")